# Patient Adherence Analytics: PySpark Implementation

This notebook re-runs the core analysis from the Python (`statsmodels`) and R (`glm`) versions using **PySpark**
(Spark DataFrames, Spark SQL window functions, and `pyspark.ml`).

**Goal:** confirm that the same logistic regression, fitted in Spark, reproduces the odds ratios from the
statsmodels and R versions.

**Why Spark here?** The dataset is small (5,000 patients), so Spark is not needed for performance. This runs in
local mode to show the same workflow (DataFrame API, Spark SQL, `VectorAssembler`, `LogisticRegression`) that
would be used on a much larger claims dataset on a cluster.

## 1. Setup

In [1]:
# In Google Colab, uncomment the next line:
# !pip install -q pyspark

import os
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("patient-adherence-pyspark")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.2.0


## 2. Load data with an explicit schema

In [2]:
# Works from the repo (notebooks/ folder), the repo root, or Colab (downloads from GitHub)
candidates = ["../data/patient_adherence.csv", "data/patient_adherence.csv"]
RAW_URL = ("https://raw.githubusercontent.com/Dhayalramesh/"
           "patient-adherence-analytics/main/data/patient_adherence.csv")
path = next((p for p in candidates if os.path.exists(p)), None)
if path is None:
    pd.read_csv(RAW_URL).to_csv("patient_adherence.csv", index=False)
    path = "patient_adherence.csv"

schema = StructType([
    StructField("patient_id", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("region", StringType(), False),
    StructField("drug_class", StringType(), False),
    StructField("prescriber_id", StringType(), False),
    StructField("copay_tier", StringType(), False),
    StructField("n_comorbidities", IntegerType(), False),
    StructField("mail_order", IntegerType(), False),
    StructField("new_to_therapy", IntegerType(), False),
    StructField("pdc", DoubleType(), False),
    StructField("non_adherent", IntegerType(), False),
    StructField("risk_cohort", StringType(), False),
])

df = spark.read.csv(path, header=True, schema=schema)
print("Rows:", df.count())
df.printSchema()
df.show(5, truncate=False)

Rows: 5000
root
 |-- patient_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- drug_class: string (nullable = true)
 |-- prescriber_id: string (nullable = true)
 |-- copay_tier: string (nullable = true)
 |-- n_comorbidities: integer (nullable = true)
 |-- mail_order: integer (nullable = true)
 |-- new_to_therapy: integer (nullable = true)
 |-- pdc: double (nullable = true)
 |-- non_adherent: integer (nullable = true)
 |-- risk_cohort: string (nullable = true)



+----------+---+---------+-----------------+-------------+----------+---------------+----------+--------------+-----+------------+-----------+
|patient_id|age|region   |drug_class       |prescriber_id|copay_tier|n_comorbidities|mail_order|new_to_therapy|pdc  |non_adherent|risk_cohort|
+----------+---+---------+-----------------+-------------+----------+---------------+----------+--------------+-----+------------+-----------+
|PT00000   |70 |Northeast|Antihypertensives|PR0024       |Medium    |1              |1         |0             |0.743|1           |Low Risk   |
|PT00001   |64 |West     |Antidiabetics    |PR0131       |Medium    |2              |0         |1             |0.502|1           |High Risk  |
|PT00002   |52 |West     |Statins          |PR0178       |Low       |1              |0         |0             |0.627|1           |Medium Risk|
|PT00003   |70 |Southeast|Statins          |PR0138       |Medium    |2              |0         |1             |0.231|1           |High Risk  |

## 3. Exploratory analysis (DataFrame API)

In [3]:
overall = df.agg(F.avg("non_adherent").alias("rate")).first()["rate"]
print(f"Overall non-adherence rate: {overall:.1%}\n")

print("By copay tier:")
(df.groupBy("copay_tier")
   .agg(F.count("*").alias("patients"),
        F.round(F.avg("non_adherent"), 3).alias("non_adherent_rate"))
   .orderBy("non_adherent_rate").show())

print("New-to-therapy vs existing:")
(df.groupBy("new_to_therapy")
   .agg(F.count("*").alias("patients"),
        F.round(F.avg("non_adherent"), 3).alias("non_adherent_rate"))
   .orderBy("new_to_therapy").show())

Overall non-adherence rate: 79.0%

By copay tier:


+----------+--------+-----------------+
|copay_tier|patients|non_adherent_rate|
+----------+--------+-----------------+
|       Low|    1922|            0.682|
|    Medium|    2076|            0.814|
|      High|    1002|            0.947|
+----------+--------+-----------------+

New-to-therapy vs existing:


+--------------+--------+-----------------+
|new_to_therapy|patients|non_adherent_rate|
+--------------+--------+-----------------+
|             0|    3018|            0.707|
|             1|    1982|            0.916|
+--------------+--------+-----------------+



## 4. Spark SQL and window functions

Region x risk-cohort summary, plus a `RANK() OVER` to order regions by non-adherence within each cohort.

In [4]:
df.createOrReplaceTempView("patients")

cohort_sql = spark.sql("""
    SELECT region,
           risk_cohort,
           COUNT(*)                          AS patients,
           ROUND(AVG(pdc), 3)                AS avg_pdc,
           ROUND(AVG(non_adherent), 3)       AS non_adherent_rate,
           RANK() OVER (PARTITION BY risk_cohort
                        ORDER BY AVG(non_adherent) DESC) AS rank_in_cohort
    FROM patients
    GROUP BY region, risk_cohort
    ORDER BY risk_cohort, rank_in_cohort
""")
cohort_sql.show(15, truncate=False)

+---------+-----------+--------+-------+-----------------+--------------+
|region   |risk_cohort|patients|avg_pdc|non_adherent_rate|rank_in_cohort|
+---------+-----------+--------+-------+-----------------+--------------+
|West     |High Risk  |151     |0.423  |0.993            |1             |
|Northeast|High Risk  |226     |0.438  |0.991            |2             |
|Southeast|High Risk  |165     |0.437  |0.982            |3             |
|Midwest  |High Risk  |206     |0.444  |0.981            |4             |
|Southwest|High Risk  |243     |0.45   |0.967            |5             |
|West     |Low Risk   |61      |0.797  |0.443            |1             |
|Northeast|Low Risk   |69      |0.814  |0.406            |2             |
|Southwest|Low Risk   |111     |0.82   |0.369            |3             |
|Southeast|Low Risk   |49      |0.841  |0.347            |4             |
|Midwest  |Low Risk   |63      |0.836  |0.317            |5             |
|Midwest  |Medium Risk|753     |0.657 

In [5]:
# Cross-check against the cohort table produced by the pandas version
ref = pd.read_csv(path.replace("patient_adherence.csv", "cohort_summary.csv")) \
      if os.path.exists(path.replace("patient_adherence.csv", "cohort_summary.csv")) else None

if ref is not None:
    mine = cohort_sql.select("region", "risk_cohort", "patients", "non_adherent_rate").toPandas()
    merged = ref.merge(mine, on=["region", "risk_cohort"], suffixes=("_pandas", "_spark"))
    same_counts = (merged["patients_pandas"] == merged["patients_spark"]).all()
    same_rates = np.allclose(merged["non_adherent_rate_pandas"], merged["non_adherent_rate_spark"], atol=1e-3)
    print(f"Cohort table rows compared: {len(merged)}")
    print(f"Patient counts identical: {same_counts} | Non-adherence rates match (tol 0.001): {same_rates}")
else:
    print("cohort_summary.csv not found next to the data file; skipping cross-check.")

Cohort table rows compared: 15
Patient counts identical: True | Non-adherence rates match (tol 0.001): True


## 5. Feature preparation and logistic regression (`pyspark.ml`)

Same model specification as the statsmodels and R versions:

`non_adherent ~ age + copay_medium + copay_high + mail_order + n_comorbidities + new_to_therapy`

`regParam=0.0` gives an unregularized fit, so it is directly comparable to the maximum-likelihood
estimates from statsmodels and `glm`.

In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

model_df = (df
    .withColumn("copay_medium", (F.col("copay_tier") == "Medium").cast("int"))
    .withColumn("copay_high",   (F.col("copay_tier") == "High").cast("int"))
    .withColumn("label",        F.col("non_adherent").cast("double")))

FEATURES = ["age", "copay_medium", "copay_high", "mail_order", "n_comorbidities", "new_to_therapy"]
assembler = VectorAssembler(inputCols=FEATURES, outputCol="features")
data = assembler.transform(model_df).select("features", "label")

lr = LogisticRegression(featuresCol="features", labelCol="label",
                        regParam=0.0, elasticNetParam=0.0,
                        maxIter=500, tol=1e-10, standardization=True)
lr_model = lr.fit(data)

coefs = np.array(lr_model.coefficients)
odds = pd.DataFrame({
    "term": ["Intercept"] + FEATURES,
    "spark_odds_ratio": np.exp(np.concatenate([[lr_model.intercept], coefs])).round(3),
})
odds

,term,spark_odds_ratio
0,Intercept,8.764
1,age,0.961
2,copay_medium,2.417
3,copay_high,13.812
4,mail_order,0.243
5,n_comorbidities,1.739
6,new_to_therapy,7.207


## 6. Compare with the statsmodels (Python) and glm (R) odds ratios

In [7]:
def find(name):
    for base in ["../", "", "./"]:
        p = os.path.join(base, name)
        if os.path.exists(p):
            return p
    return None

py_path = find("data/logistic_regression_odds_ratios.csv")
r_path = find("r/logistic_regression_odds_ratios_R.csv")

comp = odds.copy()
if py_path:
    py = pd.read_csv(py_path).rename(columns={"Unnamed: 0": "term"})[["term", "odds_ratio"]]
    comp = comp.merge(py.rename(columns={"odds_ratio": "statsmodels_or"}), on="term", how="left")
    comp["abs_diff_vs_statsmodels"] = (comp["spark_odds_ratio"] - comp["statsmodels_or"]).abs().round(3)
if r_path:
    r = pd.read_csv(r_path)[["term", "odds_ratio"]]
    r["term"] = r["term"].replace({"(Intercept)": "Intercept"})
    comp = comp.merge(r.rename(columns={"odds_ratio": "r_glm_or"}), on="term", how="left")
    comp["abs_diff_vs_r"] = (comp["spark_odds_ratio"] - comp["r_glm_or"]).abs().round(3)
comp

,term,spark_odds_ratio,statsmodels_or,abs_diff_vs_statsmodels,r_glm_or,abs_diff_vs_r
0,Intercept,8.764,8.764,0.0,8.764,0.0
1,age,0.961,0.961,0.0,0.961,0.0
2,copay_medium,2.417,2.417,0.0,2.417,0.0
3,copay_high,13.812,13.812,0.0,13.812,0.0
4,mail_order,0.243,0.243,0.0,0.243,0.0
5,n_comorbidities,1.739,1.739,0.0,1.739,0.0
6,new_to_therapy,7.207,7.207,0.0,7.207,0.0


In [8]:
if py_path:
    print("Spark vs statsmodels: agree within 0.01 on every term:", (comp["abs_diff_vs_statsmodels"] <= 0.01).all())
if r_path:
    print("Spark vs R glm:       agree within 0.01 on every term:", (comp["abs_diff_vs_r"] <= 0.01).all())

Spark vs statsmodels: agree within 0.01 on every term: True
Spark vs R glm:       agree within 0.01 on every term: True


## 7. Held-out evaluation

The comparison above uses the full dataset, like the other two implementations. Below, an 80/20 split with
3-fold cross-validation on the training portion gives a check on out-of-sample discrimination.

In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

train, test = data.randomSplit([0.8, 0.2], seed=42)

lr_cv = LogisticRegression(featuresCol="features", labelCol="label", maxIter=200)
grid = ParamGridBuilder().addGrid(lr_cv.regParam, [0.0, 0.01, 0.1]).build()
auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

cv = CrossValidator(estimator=lr_cv, estimatorParamMaps=grid,
                    evaluator=auc_eval, numFolds=3, seed=42)
cv_model = cv.fit(train)

pred = cv_model.transform(test)
auc = auc_eval.evaluate(pred)
f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
print(f"Train rows: {train.count()} | Test rows: {test.count()}")
print(f"Test AUC: {auc:.3f} | F1: {f1:.3f} | Accuracy: {acc:.3f}")
print("Best regParam from CV:", cv_model.bestModel.getRegParam())

Train rows: 4042 | Test rows: 958
Test AUC: 0.830 | F1: 0.812 | Accuracy: 0.829
Best regParam from CV: 0.0


## 8. Save results

In [10]:
out_dir = "../data" if os.path.isdir("../data") else "."
out_path = os.path.join(out_dir, "pyspark_logistic_regression_odds_ratios.csv")
comp.to_csv(out_path, index=False)
print("Saved:", out_path)
spark.stop()

Saved: ../data/pyspark_logistic_regression_odds_ratios.csv


## Summary

- Loaded the patient adherence dataset into Spark with an explicit schema.
- Reproduced the region x risk-cohort summary using Spark SQL, including a `RANK() OVER` window function.
- Fitted the same logistic regression with `pyspark.ml`. The odds ratios match the statsmodels and R results,
  so the finding holds across three independent implementations: **a high copay tier and new-to-therapy status
  are the strongest drivers of non-adherence**.
- Ran on a local Spark session. The dataset is small, so this demonstrates the workflow, not big-data scale.